# Модуль 06 · Урок 3. Потоковое чтение CSV и обработка пакетами

**Как проходить:** изучите все разделы теории последовательно. Запустите демонстрационные ячейки, осознайте входы и выходы методов. Только после этого решайте задания. В условиях используются другие предметные области и комбинации действий — готовых решений в теории нет.

**Обещание урока:** все необходимые операции объяснены ниже; внешний поиск не нужен.

**Стандарт проверки:** правильные значения, форма, порядок и неизменность входа; минимум два набора данных на задачу.

In [ ]:
# Настройка путей для работы из любой папки модуля в VS Code
from pathlib import Path
import os
import sys

ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents]
     if (p / "tests" / "notebook_checks.py").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError(
        "Не найден корень pandas_academy. Откройте папку курса целиком "
        "через File -> Open Folder в VS Code."
    )

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)  # Теперь data/ и output/ всегда находятся в корне курса

import pandas as pd
import numpy as np
from tests.notebook_checks import expected_for, check_result, show_progress, show_progress

(ROOT / "output").mkdir(exist_ok=True)
print("pandas:", pd.__version__, "| корень проекта:", ROOT)


## 1. Что такое чанки
Если CSV занимает несколько гигабайт, чтение целиком через `pd.read_csv` может превысить доступную память. Параметр `chunksize=100_000` возвращает итератор по частям — DataFrame на каждую порцию. Можно рассчитывать аддитивные показатели и хранить лишь промежуточные суммы, не склеивая все чанки в одну гигантскую таблицу.

In [ ]:
import pandas as pd
from io import StringIO
stream = StringIO('region,value\nA,2\nB,5\nA,7\n')
acc = 0
for chunk in pd.read_csv(stream,chunksize=2):
    acc += chunk.value.sum()
print(acc)

## 2. Какие агрегаты складываются, а какие нет
Сумму можно получить как сумму сумм чанков. Количество — как сумму количеств. Среднее нельзя считать средним **средних чанков**, если размеры разные; нужно накапливать суммарную сумму и количество, затем делить. Для точного количества уникальных пользователей недостаточно сложить `nunique` чанков: один пользователь может встречаться в нескольких пакетах, поэтому требуется объединить множества ID или использовать специальный алгоритм.

## 3. dtype, usecols и безопасное чтение
`usecols=['id','price']` исключает ненужные данные при чтении. `dtype={'id':'string'}` сохраняет текстовые идентификаторы, особенно ведущие нули. `parse_dates` преобразует временные поля на входе, но при неоднозначном формате лучше явная обработка. Проверяйте разделитель и кодировку; не считайте CSV «простым», пока не проверены кавычки, запятые в тексте и пустые значения.

In [ ]:
from io import StringIO
stream=StringIO('code;qty\n001;5\n020;7\n')
print(pd.read_csv(stream,sep=';',dtype={'code':'string'}))

## 4. Границы масштабирования pandas
Пакеты помогают, если данные можно агрегировать постепенно. Если нужно глобально сортировать всё множество или выполнять множество join по огромным таблицам, одна обработка чанками может не решить проблему. Тогда следует задуматься об SQL, DuckDB, Polars или Dask. Важнее понимать характер операции и предел RAM, чем выбирать библиотеку по популярности.

**Проверьте себя:** почему среднее средних неверно? Можно ли складывать `nunique` по чанкам? Почему `dtype='string'` важен для ID?

---
## Самостоятельная практика

**Инструкция:** выполните ячейку исходных данных, напишите обычный код в следующей ячейке и сохраните итог в `result`. Под решением показан ожидаемый вывод. Затем запустите проверку. **Не нужно писать функции.**

### m6_11 · read selected

Прочитайте CSV только с колонками `order_id`,`qty` (именно в этом порядке), ID должны остаться строками.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.


**Выполнение:** используйте готовые переменные `path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_11: используются переменные path

path = ROOT / "output" / "examples" / "m6_11_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
path_table.to_csv(path, index=False, encoding="utf-8")


# Снимок правильного результата для автопроверки.
_expected_m6_11 = expected_for("m6_11", path=path)


In [ ]:
# m6_11: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 6 строк, 2 столбца; колонки: `order_id`, `qty`.

```text
  order_id  qty
0  demo_o1    2
1  demo_o2    1
2  demo_o3    4
3  demo_o4    1
4  demo_o5    3
5  demo_o6    0
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_11", result, _expected_m6_11)

### m6_12 · chunked sum

Считайте CSV чанками по `size` строк и получите сумму `price * qty` без скидки как число. Не собирайте все чанки через concat.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.
- `size` — размер блока чтения CSV (`int`); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `path`, `size`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_12: используются переменные path, size

path = ROOT / "output" / "examples" / "m6_12_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
path_table.to_csv(path, index=False, encoding="utf-8")

size = 2


# Снимок правильного результата для автопроверки.
_expected_m6_12 = expected_for("m6_12", path=path, size=size)


In [ ]:
# m6_12: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path, size
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`float`. Результат для показанного примера: `1377.0`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_12", result, _expected_m6_12)

### m6_13 · chunked mean

Считайте CSV чанками и посчитайте общее среднее price, накопив сумму и число непустых значений (а не усредняя средние чанков).

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.
- `size` — размер блока чтения CSV (`int`); в этом примере: `4`.


**Выполнение:** используйте готовые переменные `path`, `size`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_13: используются переменные path, size

path = ROOT / "output" / "examples" / "m6_13_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
path_table.to_csv(path, index=False, encoding="utf-8")

size = 4


# Снимок правильного результата для автопроверки.
_expected_m6_13 = expected_for("m6_13", path=path, size=size)


In [ ]:
# m6_13: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path, size
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`float`. Результат для показанного примера: `165.33333333333334`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_13", result, _expected_m6_13)

### m6_14 · chunked unique

Получите число уникальных client_id во всём CSV при чтении чанками. Повторяющиеся между чанками ID считать один раз.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.
- `size` — размер блока чтения CSV (`int`); в этом примере: `2`.


**Выполнение:** используйте готовые переменные `path`, `size`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_14: используются переменные path, size

path = ROOT / "output" / "examples" / "m6_14_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
path_table.to_csv(path, index=False, encoding="utf-8")

size = 2


# Снимок правильного результата для автопроверки.
_expected_m6_14 = expected_for("m6_14", path=path, size=size)


In [ ]:
# m6_14: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path, size
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`int`. Результат для показанного примера: `4`.

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_14", result, _expected_m6_14)

### m6_15 · select columns csv

Прочтите CSV и получите DataFrame только с городом, ценой и количеством; строки с quantity=0 исключите, сохранив исходный порядок и сбросив индекс.

**Исходные данные** создаются в следующей ячейке и сразу доступны по указанным именам:

- `path` — путь к файлу (`pathlib.Path`), создаётся отдельной ячейкой ниже.


**Выполнение:** используйте готовые переменные `path`. Напишите обычный код (одну или несколько строк), а итог сохраните в **`result`**. Ничего не нужно оформлять в функцию.


In [ ]:
# Исходные данные задания m6_15: используются переменные path

path = ROOT / "output" / "examples" / "m6_15_example.csv"
path.parent.mkdir(parents=True, exist_ok=True)
path_table = pd.DataFrame({
    'order_id': ['demo_o1', 'demo_o2', 'demo_o3', 'demo_o4', 'demo_o5', 'demo_o6'],
    'client_id': [1, 2, 1, 3, 2, 4],
    'city': ['Москва', 'Тверь', 'Москва', 'Казань', 'Тверь', 'Казань'],
    'category': ['чай', 'кофе', 'чай', 'книги', 'кофе', 'чай'],
    'price': [127.0, 247.0, 97.0, 317.0, 57.0, 147.0],
    'qty': [2, 1, 4, 1, 3, 0],
    'discount': [0, 10, 5, 0, 20, 0],
    'status': ['paid', 'new', 'paid', 'paid', 'cancelled', 'new'],
    'date': ['2026-01-03', '2026-01-04', '2026-01-04', '2026-02-01', '2026-02-02', '2026-02-05'],
    'comment': ['  Быстрая доставка  ', 'без звонка', 'Срочно', 'подарок', '', None],
})
path_table.to_csv(path, index=False, encoding="utf-8")


# Снимок правильного результата для автопроверки.
_expected_m6_15 = expected_for("m6_15", path=path)


In [ ]:
# m6_15: напишите решение прямо здесь. Функции создавать НЕ нужно.
# Используйте переменные: path
# Сохраните конечный результат в переменную result.
result = None  # замените этой строкой/своим кодом
result

**Ожидаемый вывод** (для данных из ячейки выше):

`pd.DataFrame`: 5 строк, 3 столбца; колонки: `city`, `price`, `qty`.

```text
     city  price  qty
0  Москва  127.0    2
1   Тверь  247.0    1
2  Москва   97.0    4
3  Казань  317.0    1
4   Тверь   57.0    3
```

In [ ]:
# Проверка результата предыдущей ячейки
check_result("m6_15", result, _expected_m6_15)

---
## Итоги урока

Если все проверки пройдены, повторите своими словами: **что делает каждый метод, какие параметры важны и какая ошибка наиболее вероятна**. Перед переходом к следующему уроку попробуйте решить одно задание заново с чистой ячейки. Автотест подтверждает поведение на известных наборах, но не заменяет код-ревью.

In [ ]:
show_progress()